In [1]:
import duckdb
import pandas as pd
from datetime import datetime

In [2]:
# Connect to the database
db = duckdb.connect("patents_training.db")

In [3]:
db.sql("SHOW TABLES")

┌────────────────────┐
│        name        │
│      varchar       │
├────────────────────┤
│ patents_embeddings │
│ patents_raw        │
└────────────────────┘

In [5]:
db.sql("SELECT * FROM patents_raw").df()

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-11540543-B2,38969498,US17036785,Sweetened consumables comprising mogroside IV ...,Disclosed are sweetened consumables and method...,"['A23L29/37', 'A23L29/30', 'A23L33/105', 'A23L...",2023,US,out,NaN,NaN,NaN,NaN,NaN,False
1,US-20210015134-A1,38969498,US17036785,CONSUMABLES,Disclosed are sweetened consumables and method...,"['A23L27/30', 'A23L2/60', 'A23V2002/00', 'A23L...",2021,US,out,NaN,NaN,NaN,NaN,NaN,False
2,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
3,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
4,EP-3542638-A1,44510082,EP19173302.1,NUTRITIONAL COMPOSITION,Non-medical use of at least two components sel...,"['A23L33/13', 'A61K33/04', 'A23L33/40', 'A61K3...",2019,EP,out,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2445,WO-2025079037-A2,99798170,IB2024/059990,"PRODUCT, SYSTEM AND METHOD OF CELL CULTIVATION",The present invention relates to a cell biomas...,"['C12P21/02', 'C12N2500/38', 'C12N2500/32', 'C...",2025,WO,in,CM,NaN,Bioprocess design,Meat,NaN,False
2446,WO-2025113823-A1,99799704,EP2024/025329,POWDER MIX FORMULATIONS WITH IMPROVED SENSORY ...,The invention relates to a powder preparation ...,"['A23L2/66', 'A23L2/39', 'A23L33/185', 'A23L33...",2025,WO,in,PB,NaN,Ingredient optimisation,Cross-cutting,"Isolates, concentrates, and flours",False
2447,US-20250236833-A1,99799723,US19028830,"Product, system and method of cell cultivation","The present invention provides products, syste...","['C12N5/0653', 'A23K40/20', 'A23L33/10', 'A23K...",2025,US,in,CM,NaN,Bioprocess design,Meat,NaN,False
2448,US-20250270503-A1,99799739,US19028930,"Product, system and method of cell cultivation","The present invention provides products, syste...","['A23K10/20', 'C12N2510/04', 'C12M33/10', 'C12...",2025,US,in,CM,NaN,Bioprocess design,Meat,NaN,False


In [4]:
# Have a look at the data stored in the csv file
db.sql("SELECT id, title, abstract, publication_year, scope, pillar, research_category  FROM 'Patents_Data/patents_training_data.csv'").show()

┌────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# This would be a way to create the table and load the data in one step, but it doesn't allow us to set the PublicationID as the primary key
# con.sql("CREATE TABLE publications_raw AS SELECT id, title, abstract, year, scope, pillar, research_category FROM 'publications_data.csv'")

# This line deletes the created table
# con.sql("DROP TABLE IF EXISTS publications_raw")

In [5]:
# In order to have the Publication ID as the primary key, we need to create the table in two steps:
# Step 1: create the table with the schema
db.sql("""
    CREATE TABLE patents_raw (
        id VARCHAR PRIMARY KEY,
        family_id INT,
        application_number VARCHAR,
        title VARCHAR,
        abstract VARCHAR,
        cpc VARCHAR,
        publication_year INT,
        jurisdiction VARCHAR,
        scope VARCHAR,
        pillar VARCHAR,
        subpillar VARCHAR,
        research_category VARCHAR,
        endproduct VARCHAR,
        ingredient VARCHAR
    )
""")

# Step 2: load the CSV into it
db.sql("INSERT INTO patents_raw SELECT id, family_id, application_number, title, abstract, cpc, " \
"publication_year, jurisdiction, scope, pillar, subpillar, research_category, endproduct, ingredient FROM 'Patents_Data/patents_training_data.csv'")

In [6]:
db.sql("DESCRIBE patents_raw").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                 │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ family_id          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ application_number │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ title              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cpc                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ publication_year   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ jurisdiction       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ scope              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [8]:
# Extract in scope patents
db.sql("""
    SELECT COUNT(*)
    FROM patents_raw
    WHERE scope = 'in'
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1280 │
└──────────────┘



In [9]:
# Convert to pandas dataframe
df = db.sql("SELECT * FROM patents_raw").df()

In [10]:
# Filter for in scope publications
df[df['scope'] == 'in']

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient
2,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN
3,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN
28,EP-4424172-A3,52462090,EP24182765.8,A PROTEINACEOUS MEAT ANALOGUE HAVING AN IMPROV...,The invention concerns an extended shelf-life ...,"['A23V2002/00', 'A23V2200/20', 'A23J3/22', 'A2...",2025,EP,in,PB,NaN,End product formulation,Meat,NaN
29,US-11666069-B2,52462090,US17530304,Proteinaceous meat analogue having an improved...,An extended shelf-life proteinaceous meat anal...,"['A23J3/22', 'A23J3/18', 'A23V2200/20', 'A23V2...",2023,US,in,PB,NaN,End product formulation,Meat,NaN
32,EP-4627930-A3,52596486,EP25169128.3,VARIANTS OF CHYMOSIN WITH IMPROVED MILK-CLOTTI...,Variants of chymosin with improved milk-clotti...,"['A23C19/04', 'A23C19/041', 'C12N9/6483', 'A23...",2026,EP,in,PB,NaN,End product formulation,Cheese,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2454,WO-2025079037-A2,99798170,IB2024/059990,"PRODUCT, SYSTEM AND METHOD OF CELL CULTIVATION",The present invention relates to a cell biomas...,"['C12P21/02', 'C12N2500/38', 'C12N2500/32', 'C...",2025,WO,in,CM,NaN,Bioprocess design,Meat,NaN
2455,WO-2025113823-A1,99799704,EP2024/025329,POWDER MIX FORMULATIONS WITH IMPROVED SENSORY ...,The invention relates to a powder preparation ...,"['A23L2/66', 'A23L2/39', 'A23L33/185', 'A23L33...",2025,WO,in,PB,NaN,Ingredient optimisation,Cross-cutting,"Isolates, concentates, and flours"
2456,US-20250236833-A1,99799723,US19028830,"Product, system and method of cell cultivation","The present invention provides products, syste...","['C12N5/0653', 'A23K40/20', 'A23L33/10', 'A23K...",2025,US,in,CM,NaN,Bioprocess design,Meat,NaN
2457,US-20250270503-A1,99799739,US19028930,"Product, system and method of cell cultivation","The present invention provides products, syste...","['A23K10/20', 'C12N2510/04', 'C12M33/10', 'C12...",2025,US,in,CM,NaN,Bioprocess design,Meat,NaN


In [15]:
# Update patents training data with manually reviewed labels
COLUMNS  = ["ingredient"]
CSV_PATH = "GenAI/ingredient/ingredient_test_data_rand4.csv"

df = pd.read_csv(CSV_PATH)[["id"] + COLUMNS]
db.register("updates", df)
set_clause = ", ".join(f"{c} = updates.{c}" for c in COLUMNS)
db.execute(f"UPDATE patents_raw SET {set_clause} FROM updates WHERE patents_raw.id = updates.id")

In [ ]:
db.close()

#### Create subset dataset for pipeline test run

In [4]:
def create_balanced_sample(df, n_out, n_pb, n_f, n_cultivated, n_cross, random_state=4):
    out_sample  = df[df["scope"] == "out"].sample(n=n_out,        random_state=random_state)
    pb_sample   = df[df["pillar"] == "PB"].sample(n=n_pb,         random_state=random_state)
    f_sample    = df[df["pillar"] == "F"].sample(n=n_f,           random_state=random_state)
    cult_sample = df[df["pillar"] == "CM"].sample(n=n_cultivated,  random_state=random_state)
    cross_sample= df[df["pillar"] == "CC"].sample(n=n_cross,       random_state=random_state)
    combined = pd.concat(
        [out_sample, pb_sample, f_sample, cult_sample, cross_sample], ignore_index=True
    )
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [7]:
# Generate artificial run table for test running the pipeline on this subset
db = duckdb.connect("patents_training.db")
df = db.sql("SELECT * FROM patents_raw").df()
db.close()

# subset training data
df_subset = create_balanced_sample(df, 250, 200, 150, 120, 100)

# save as csv
df_subset.to_csv("Patents_Data/patents_subest_for_pipeline_testing2.csv")

In [8]:
# only columns that would have come out of a dimensions query
df_subset = df_subset.loc[:, "id":"jurisdiction"]
df_subset["date_dimensions"] = datetime.today().strftime('%y%m%d')

# save to patents.db as artificial run table
db = duckdb.connect("../Pipeline/Patents/patents.db")
db.sql("CREATE TABLE run_260710_0000 AS SELECT * FROM df_subset")
db.close()